In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import gc
from IPython.display import display

# =========================
# CONFIG
# =========================

def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook working directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Run this notebook inside the project directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "CIC-ToN-IoT.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "metrics" / "feature_check_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Set this if the label column name is already known.
# Leave it as None to detect the label column automatically.
LABEL_COL = None

# Keep this conservative for large datasets.
SAMPLE_SIZE = 100_000
CHUNK_SIZE = 100_000

MAX_MISSING_PERCENT = 30
MAX_INF_PERCENT = 30
QUASI_CONSTANT_THRESHOLD = 0.99

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)


In [ ]:
def read_csv_sample(path, nrows=100_000):
    encodings = ["utf-8", "latin1", "ISO-8859-1"]
    last_error = None
    
    for enc in encodings:
        try:
            sample = pd.read_csv(
                path,
                nrows=nrows,
                encoding=enc,
                low_memory=False
            )
            return sample, enc
        except Exception as e:
            last_error = e
    
    raise RuntimeError(f"Failed to read CSV sample. Last error: {last_error}")


sample_df, USED_ENCODING = read_csv_sample(DATA_PATH, SAMPLE_SIZE)

print("Sample loaded successfully.")
print("Encoding used:", USED_ENCODING)
print("Sample shape:", sample_df.shape)

display(sample_df.head())

In [ ]:
print("Total columns:", len(sample_df.columns))
print("\nColumns:")
for i, col in enumerate(sample_df.columns, start=1):
    print(f"{i}. {col}")

possible_label_keywords = ["label", "class", "attack", "category", "type", "target"]

possible_label_cols = [
    col for col in sample_df.columns
    if any(keyword in col.lower() for keyword in possible_label_keywords)
]

print("\nPossible label columns:")
print(possible_label_cols)

for col in possible_label_cols:
    print("\n" + "=" * 60)
    print("Column:", col)
    print("Unique values in sample:", sample_df[col].nunique(dropna=False))
    print(sample_df[col].value_counts(dropna=False).head(30))

In [ ]:
def auto_select_label_column(df, possible_cols):
    if not possible_cols:
        return df.columns[-1]
    
    candidates = []
    
    for col in possible_cols:
        unique_count = df[col].nunique(dropna=False)
        name = col.lower()
        
        score = 0
        
        if "attack" in name:
            score += 5
        if "label" in name:
            score += 4
        if "class" in name:
            score += 3
        if "category" in name or "type" in name:
            score += 2
        
        # For multiclass data, prefer columns with 3 to 50 classes.
        if 3 <= unique_count <= 50:
            score += 10
        elif unique_count == 2:
            score += 3
        
        candidates.append((score, col, unique_count))
    
    candidates = sorted(candidates, reverse=True)
    return candidates[0][1]


if LABEL_COL is None:
    LABEL_COL = auto_select_label_column(sample_df, possible_label_cols)

print("Selected label column:", LABEL_COL)

if LABEL_COL not in sample_df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found. Please check the column name.")

print("\nSample class distribution:")
display(sample_df[LABEL_COL].value_counts(dropna=False))

In [ ]:
X_sample = sample_df.drop(columns=[LABEL_COL])
y_sample = sample_df[LABEL_COL]

def numeric_convert_ratio(series):
    non_null = series.dropna()
    
    if len(non_null) == 0:
        return 0.0
    
    if pd.api.types.is_numeric_dtype(non_null):
        return 1.0
    
    converted = pd.to_numeric(non_null, errors="coerce")
    return converted.notna().mean()


feature_basic = []

for col in X_sample.columns:
    s = X_sample[col]
    
    numeric_ratio = numeric_convert_ratio(s)
    is_numeric_candidate = numeric_ratio >= 0.95
    
    unique_count = s.nunique(dropna=False)
    unique_ratio = unique_count / max(len(s), 1)
    
    value_counts = s.value_counts(dropna=False, normalize=True)
    top_value_frequency = value_counts.iloc[0] if len(value_counts) > 0 else 1.0
    
    feature_basic.append({
        "feature": col,
        "sample_dtype": str(s.dtype),
        "sample_unique_count": unique_count,
        "sample_unique_ratio": unique_ratio,
        "sample_top_value_frequency": top_value_frequency,
        "numeric_convert_ratio": numeric_ratio,
        "is_numeric_candidate": is_numeric_candidate
    })

feature_basic_df = pd.DataFrame(feature_basic)

display(feature_basic_df.head())

In [ ]:
def is_suspected_identifier(col):
    name = col.lower().strip()
    
    exact_bad = [
        "id", "flow id", "flow_id", "uid",
        "src ip", "src_ip", "source ip", "source_ip",
        "dst ip", "dst_ip", "destination ip", "destination_ip",
        "timestamp", "time stamp", "date"
    ]
    
    if name in exact_bad:
        return True
    
    bad_patterns = [
        "src ip", "dst ip", "source ip", "destination ip",
        "src_ip", "dst_ip", "source_ip", "destination_ip",
        "timestamp"
    ]
    
    if any(pattern in name for pattern in bad_patterns):
        return True
    
    # Do not automatically drop features containing the word "flow",
    # because features such as Flow Duration can still be valid.
    return False


suspected_identifier_cols = [
    col for col in X_sample.columns
    if is_suspected_identifier(col)
]

print("Suspected identifier or leakage-prone columns:")
for col in suspected_identifier_cols:
    print("-", col)

In [ ]:
all_columns = sample_df.columns.tolist()
feature_columns = [col for col in all_columns if col != LABEL_COL]

numeric_candidate_cols = feature_basic_df.loc[
    feature_basic_df["is_numeric_candidate"],
    "feature"
].tolist()

missing_counts = Counter()
inf_counts = Counter()
total_rows = 0
class_counts = Counter()

print("Start chunk profiling...")

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding=USED_ENCODING,
    low_memory=False
)

for chunk_idx, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)
    
    # Class distribution
    if LABEL_COL in chunk.columns:
        class_counts.update(chunk[LABEL_COL].value_counts(dropna=False).to_dict())
    
    # Missing values
    missing_counts.update(chunk[feature_columns].isna().sum().to_dict())
    
    # Check infinite values only for candidate numeric features.
    for col in numeric_candidate_cols:
        if col in chunk.columns:
            numeric_col = pd.to_numeric(chunk[col], errors="coerce")
            inf_counts[col] += np.isinf(numeric_col).sum()
    
    if chunk_idx % 10 == 0:
        print(f"Processed chunks: {chunk_idx}, rows: {total_rows:,}")
    
    del chunk
    gc.collect()

print("Chunk profiling finished.")
print("Total rows:", total_rows)

In [ ]:
feature_summary = feature_basic_df.copy()

feature_summary["missing_count"] = feature_summary["feature"].map(lambda x: missing_counts.get(x, 0))
feature_summary["missing_percent"] = feature_summary["missing_count"] / total_rows * 100

feature_summary["inf_count"] = feature_summary["feature"].map(lambda x: inf_counts.get(x, 0))
feature_summary["inf_percent"] = feature_summary["inf_count"] / total_rows * 100

feature_summary["suspected_identifier"] = feature_summary["feature"].isin(suspected_identifier_cols)

feature_summary["is_quasi_constant"] = (
    feature_summary["sample_top_value_frequency"] >= QUASI_CONSTANT_THRESHOLD
)

feature_summary["recommended_drop_reason"] = ""

def add_reason(row):
    reasons = []
    
    if not row["is_numeric_candidate"]:
        reasons.append("non numeric or not safely convertible to numeric")
    
    if row["suspected_identifier"]:
        reasons.append("identifier or leakage prone column")
    
    if row["missing_percent"] > MAX_MISSING_PERCENT:
        reasons.append(f"missing values > {MAX_MISSING_PERCENT}%")
    
    if row["inf_percent"] > MAX_INF_PERCENT:
        reasons.append(f"infinite values > {MAX_INF_PERCENT}%")
    
    if row["is_quasi_constant"]:
        reasons.append("quasi constant feature")
    
    return "; ".join(reasons)

feature_summary["recommended_drop_reason"] = feature_summary.apply(add_reason, axis=1)

feature_summary["recommended_for_autoencoder"] = feature_summary["recommended_drop_reason"] == ""

feature_summary = feature_summary.sort_values(
    by=["recommended_for_autoencoder", "recommended_drop_reason", "feature"],
    ascending=[False, True, True]
)

display(feature_summary)

feature_summary.to_csv(OUTPUT_DIR / "feature_summary.csv", index=False)
print("Saved:", OUTPUT_DIR / "feature_summary.csv")

In [ ]:
candidate_features = feature_summary.loc[
    feature_summary["recommended_for_autoencoder"],
    "feature"
].tolist()

dropped_features = feature_summary.loc[
    ~feature_summary["recommended_for_autoencoder"],
    ["feature", "recommended_drop_reason"]
]

print("Total recommended features for Autoencoder:", len(candidate_features))
print("\nRecommended features:")
for i, col in enumerate(candidate_features, start=1):
    print(f"{i}. {col}")

print("\nDropped or excluded features:")
display(dropped_features)

In [ ]:
# Save candidate features
with open(OUTPUT_DIR / "candidate_features.txt", "w", encoding="utf-8") as f:
    for col in candidate_features:
        f.write(col + "\n")

# Save dropped features
dropped_features.to_csv(OUTPUT_DIR / "dropped_features.csv", index=False)

# Save class distribution
class_distribution = pd.DataFrame({
    "class": list(class_counts.keys()),
    "count": list(class_counts.values())
})

class_distribution["percentage"] = class_distribution["count"] / class_distribution["count"].sum() * 100
class_distribution = class_distribution.sort_values(by="count", ascending=False)

display(class_distribution)

class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)

print("Saved candidate_features.txt")
print("Saved dropped_features.csv")
print("Saved class_distribution.csv")

In [ ]:
min_class_count = class_distribution["count"].min()
num_classes = class_distribution.shape[0]

print("Number of classes:", num_classes)
print("Minimum class count:", min_class_count)

if min_class_count < 2:
    print("Warning: At least one class has fewer than 2 samples. Stratified split may fail.")
else:
    print("Stratified split should be possible.")

In [ ]:
report_path = OUTPUT_DIR / "feature_check_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("FEATURE CHECK REPORT\n")
    f.write("====================\n\n")
    
    f.write(f"Dataset path: {DATA_PATH}\n")
    f.write(f"Total rows: {total_rows}\n")
    f.write(f"Total columns: {len(all_columns)}\n")
    f.write(f"Selected label column: {LABEL_COL}\n")
    f.write(f"Number of classes: {num_classes}\n\n")
    
    f.write("Class distribution:\n")
    for _, row in class_distribution.iterrows():
        f.write(f"{row['class']}: {row['count']} ({row['percentage']:.4f}%)\n")
    
    f.write("\n")
    f.write(f"Numeric candidate columns: {len(numeric_candidate_cols)}\n")
    f.write(f"Suspected identifier columns: {len(suspected_identifier_cols)}\n")
    f.write(f"Recommended Autoencoder features: {len(candidate_features)}\n")
    f.write(f"Dropped or excluded features: {len(dropped_features)}\n\n")
    
    f.write("Recommended features for Autoencoder:\n")
    for col in candidate_features:
        f.write(f"- {col}\n")
    
    f.write("\nDropped or excluded features:\n")
    for _, row in dropped_features.iterrows():
        f.write(f"- {row['feature']}: {row['recommended_drop_reason']}\n")

print("Saved report:", report_path)